In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/osb_dryer_shifts.csv")

In [2]:
df.head()

,Date,Shift,Dryer_ID,Planned_Time_Min,Downtime_Min,Blocked_Starved_Min,Ideal_Rate_BDT_Hr,Output_BDT,Rework_BDT,Fines_BDT
0,2026-01-01,Day,DRYER_1,720,93,0,26.0,240.02,8.61,2.01
1,2026-01-01,Day,DRYER_2,720,97,0,29.0,255.68,8.96,3.84
2,2026-01-01,Night,DRYER_1,720,240,0,26.0,174.30,7.43,1.34
3,2026-01-01,Night,DRYER_2,720,50,86,29.0,237.77,8.13,2.48
4,2026-01-02,Day,DRYER_1,720,27,0,26.0,253.64,6.62,3.55


In [3]:
df.dtypes

Date                       str
Shift                      str
Dryer_ID                   str
Planned_Time_Min         int64
Downtime_Min             int64
Blocked_Starved_Min      int64
Ideal_Rate_BDT_Hr      float64
Output_BDT             float64
Rework_BDT             float64
Fines_BDT              float64
dtype: object

In [4]:
df.shape

(724, 10)

In [5]:
df['Shift'].unique()

<StringArray>
['Day', 'Night']
Length: 2, dtype: str

In [6]:
df['Dryer_ID'].unique()

<StringArray>
['DRYER_1', 'DRYER_2']
Length: 2, dtype: str

In [7]:
df['Planned_Time_Min'].unique()

array([720, 240, 480])

In [8]:
df['Run_Time'] = df['Planned_Time_Min'] - df['Downtime_Min'] - df['Blocked_Starved_Min']

In [9]:
df[['Planned_Time_Min','Downtime_Min','Blocked_Starved_Min','Run_Time']].head()

,Planned_Time_Min,Downtime_Min,Blocked_Starved_Min,Run_Time
0,720,93,0,627
1,720,97,0,623
2,720,240,0,480
3,720,50,86,584
4,720,27,0,693


In [10]:
df['Run_Time'].describe()

count    724.000000
mean     597.707182
std      120.351226
min       66.000000
25%      568.000000
50%      636.000000
75%      679.250000
max      718.000000
Name: Run_Time, dtype: float64

In [11]:
df.isna().sum()

Date                   0
Shift                  0
Dryer_ID               0
Planned_Time_Min       0
Downtime_Min           0
Blocked_Starved_Min    0
Ideal_Rate_BDT_Hr      0
Output_BDT             0
Rework_BDT             0
Fines_BDT              0
Run_Time               0
dtype: int64

In [12]:
df.loc[df['Run_Time'].idxmin()]

Date                   2026-01-11
Shift                         Day
Dryer_ID                  DRYER_2
Planned_Time_Min              240
Downtime_Min                  174
Blocked_Starved_Min             0
Ideal_Rate_BDT_Hr            29.0
Output_BDT                   27.8
Rework_BDT                   0.69
Fines_BDT                    0.23
Run_Time                       66
Name: 41, dtype: object

In [13]:
df['Availability'] = df['Run_Time'] / df['Planned_Time_Min']
df['Performance'] = df['Output_BDT'] / (df['Ideal_Rate_BDT_Hr'] * (df['Run_Time'] / 60))
df['Quality'] = (df['Output_BDT'] - (df['Rework_BDT'] + df['Fines_BDT'])) / df['Output_BDT']
df['OEE'] = df['Availability'] * df['Performance'] * df['Quality']

In [14]:
df[['Availability','Performance','Quality','OEE']].describe()

,Availability,Performance,Quality,OEE
count,724.000000,724.000000,724.000000,724.000000
mean,0.866070,0.888279,0.959833,0.738847
std,0.115126,0.030602,0.008831,0.104322
min,0.250000,0.794805,0.930812,0.210321
25%,0.823264,0.867377,0.953859,0.692205
50%,0.897222,0.886490,0.959836,0.761458
75%,0.948611,0.909684,0.965812,0.810172
max,1.000000,0.980449,0.983156,0.908621


In [15]:
df['Ideal_BDT'] = df['Ideal_Rate_BDT_Hr'] * (df['Run_Time'] / 60)

In [16]:
summary = df.groupby('Dryer_ID')[
  ['Run_Time', 'Planned_Time_Min', 'Output_BDT', 'Rework_BDT', 'Fines_BDT', 'Ideal_BDT']
].sum()

In [17]:
summary['Availability'] = summary['Run_Time'] / summary['Planned_Time_Min']
summary['Performance'] = summary['Output_BDT'] / summary['Ideal_BDT']
summary['Quality'] = (summary['Output_BDT'] - (summary['Rework_BDT'] + summary['Fines_BDT'])) / summary['Output_BDT']
summary['OEE'] = summary['Availability'] * summary['Performance'] * summary['Quality']

In [18]:
summary[['Availability','Performance','Quality','OEE']]

,Availability,Performance,Quality,OEE
Dryer_ID,,,,
DRYER_1,0.855732,0.879911,0.959220,0.722262
DRYER_2,0.881342,0.896315,0.960242,0.758553


In [19]:
summary_shift = df.groupby(['Dryer_ID','Shift'])[['Run_Time', 'Planned_Time_Min','Output_BDT', 'Rework_BDT', 'Fines_BDT', 'Ideal_BDT']].sum()

In [20]:
summary_shift['Availability'] = summary_shift['Run_Time'] / summary_shift['Planned_Time_Min']
summary_shift['Performance'] = summary_shift['Output_BDT'] / summary_shift['Ideal_BDT']
summary_shift['Quality'] = (summary_shift['Output_BDT'] - (summary_shift['Rework_BDT'] + summary_shift['Fines_BDT'])) / summary_shift['Output_BDT']
summary_shift['OEE'] = summary_shift['Availability'] * summary_shift['Performance'] * summary_shift['Quality']

In [21]:
summary_shift[['Availability','Performance','Quality','OEE']]


Availability  Performance   Quality       OEE
Dryer_ID Shift                                               
DRYER_1  Day        0.858103     0.898535  0.958967  0.739398
         Night      0.853468     0.862032  0.959472  0.705900
DRYER_2  Day        0.886284     0.895630  0.961009  0.762832
         Night      0.876624     0.896977  0.959502  0.754467

In [22]:
df['Date'] = pd.to_datetime(df['Date'])

In [23]:
df.dtypes

Date                   datetime64[us]
Shift                             str
Dryer_ID                          str
Planned_Time_Min                int64
Downtime_Min                    int64
Blocked_Starved_Min             int64
Ideal_Rate_BDT_Hr             float64
Output_BDT                    float64
Rework_BDT                    float64
Fines_BDT                     float64
Run_Time                        int64
Availability                  float64
Performance                   float64
Quality                       float64
OEE                           float64
Ideal_BDT                     float64
dtype: object

In [24]:
df['Month'] = df['Date'].dt.to_period('M')

In [25]:
d1 = df[df['Dryer_ID'] == 'DRYER_1']

In [26]:
DRYER_1 = d1.groupby(['Month', 'Shift'])[['Output_BDT', 'Ideal_BDT']].sum()

In [27]:
DRYER_1['Performance'] = DRYER_1['Output_BDT'] / DRYER_1['Ideal_BDT']
DRYER_1

Output_BDT    Ideal_BDT  Performance
Month   Shift                                      
2026-01 Day       7164.44  8158.800000     0.878124
        Night     7089.99  8152.300000     0.869692
2026-02 Day       5894.56  6727.500000     0.876189
        Night     6557.35  7592.433333     0.863669
2026-03 Day       7173.04  7969.433333     0.900069
        Night     6927.22  8041.366667     0.861448
2026-04 Day       6971.16  7750.600000     0.899435
        Night     7009.47  8113.300000     0.863948
2026-05 Day       6336.31  6874.833333     0.921667
        Night     6122.96  7107.100000     0.861527
2026-06 Day       7115.66  7764.900000     0.916388
        Night     6922.26  8125.433333     0.851925

In [28]:
DRYER_1[['Output_BDT', 'Ideal_BDT']]

Output_BDT    Ideal_BDT
Month   Shift                         
2026-01 Day       7164.44  8158.800000
        Night     7089.99  8152.300000
2026-02 Day       5894.56  6727.500000
        Night     6557.35  7592.433333
2026-03 Day       7173.04  7969.433333
        Night     6927.22  8041.366667
2026-04 Day       6971.16  7750.600000
        Night     7009.47  8113.300000
2026-05 Day       6336.31  6874.833333
        Night     6122.96  7107.100000
2026-06 Day       7115.66  7764.900000
        Night     6922.26  8125.433333

## Finding 1 — DRYER_1 night shift performance gap

**What the data shows**

DRYER_1 night shift performance is flat at ~0.86 across all six months, while day
shift on the same dryer climbed from 0.878 to 0.916. The gap widens steadily:

| Month | Day | Night | Gap |
|---|---|---|---|
| 2026-01 | 0.878 | 0.870 | 0.8 pts |
| 2026-02 | 0.876 | 0.864 | 1.2 |
| 2026-03 | 0.900 | 0.861 | 3.9 |
| 2026-04 | 0.899 | 0.864 | 3.5 |
| 2026-05 | 0.922 | 0.862 | 6.0 |
| 2026-06 | 0.916 | 0.852 | 6.4 |

**What it rules out**

- Not availability — flat across both shifts (0.858 Day vs 0.853 Night).
- Not quality — flat across both shifts (~0.96).
- Not equipment wear — day shift improved 4.4 points on the same dryer over the
  same period. Equipment condition can't be selective by time of day.

In June, night shift actually ran *more* minutes than day shift (8125 vs 7765
ideal BDT) and still produced less. The dryer was available; it wasn't moving
material at rate.

**What it points to**

A procedural or setpoint difference that day shift adopted and night shift did
not. The gap was near zero in January, so this is a change that happened during
the period rather than a standing condition.

**Recommended next step**

Compare drying temperature and infeed rate setpoints between shifts before
scheduling any maintenance. This is a knowledge-transfer question, not a
mechanical one.

In [29]:
ev = pd.read_csv("data/osb_downtime_events.csv")
ev.head()
ev.shape
ev.dtypes
ev['Category'].unique()


<StringArray>
[    'Process',  'Electrical',  'Changeover',  'Mechanical',     'Blocked',
     'Starved', 'Fire/Safety']
Length: 7, dtype: str

In [30]:
ev.head()

,Event_ID,Date,Shift,Dryer_ID,Reason,Category,Duration_Min
0,1,2026-01-01,Day,DRYER_1,Dryer plug / material buildup,Process,35
1,2,2026-01-01,Day,DRYER_1,Dryer plug / material buildup,Process,15
2,3,2026-01-01,Day,DRYER_2,Wet bin plug,Process,25
3,4,2026-01-01,Day,DRYER_2,Moisture control out of spec,Process,30
4,5,2026-01-01,Day,DRYER_2,Moisture control out of spec,Process,20


In [31]:
ev.shape

(1718, 7)

In [32]:
ev.dtypes

Event_ID        int64
Date              str
Shift             str
Dryer_ID          str
Reason            str
Category          str
Duration_Min    int64
dtype: object

In [33]:
ev.groupby('Category')['Duration_Min'].sum().sort_values(ascending=False)

Category
Process        17890
Blocked        11028
Starved         9212
Mechanical      8090
Electrical      4050
Changeover      2200
Fire/Safety     2045
Name: Duration_Min, dtype: int64

In [34]:
ev['Is_Dryer_Fault'] = ~ev['Category'].isin(['Blocked','Starved'])